# Build the SFINCS test-case models: regular 100 m + subgrid 200 m / 400 m

Builds (does **not** run) SFINCS models for the three generalisation test splits:

| Split | DEMs | Rainfall |
|---|---|---|
| `test_unseen_rainfall` | 30 seen 64x64 (flat + sloped) | RP100 |
| `test_unseen_domain` | 16 unseen 128x128 (flat + sloped) | RP002, RP005, RP010, RP020, RP050 |
| `test_unseen_domain_and_rainfall` | 16 unseen 128x128 | RP100 |

in three variants:

| Variant | Compute grid | Subgrid |
|---|---|---|
| `regular_100m` | 100 m | none (the original setup from `sfincs.py`) |
| `subgrid_200m` | 200 m | 25 m pixels (8x8 per cell) |
| `subgrid_400m` | 400 m | 25 m pixels (16x16 per cell) |

Environment: `environment_rebuild.yml` (HydroMT-SFINCS 1.x).

### Imports and paths

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # notebooks/sfincs_rebuild -> repository root
sys.path.insert(0, str(PROJECT_ROOT))

from hydromt_sfincs import SfincsModel
from scripts.sfincs_rebuild.cases import build_case_table
from scripts.sfincs_rebuild.build import VARIANTS, build_all

MODEL_ROOT = PROJECT_ROOT / "models" / "rebuild"
OUTPUT_FOLDER = PROJECT_ROOT / "outputs" / "rebuild"
TIF_FOLDER = OUTPUT_FOLDER / "dem_tif"

## Case table

In [ ]:
case_table = build_case_table(PROJECT_ROOT, OUTPUT_FOLDER / "case_table.csv")
print(len(case_table), "cases")
print(case_table["split"].value_counts())
case_table.head()

## Smoke test: one case per split, all variants

In [ ]:
smoke_cases = case_table.groupby("split").head(1)
smoke_summary = build_all(smoke_cases, PROJECT_ROOT, MODEL_ROOT, TIF_FOLDER, overwrite=True)
smoke_summary

## Build everything

In [ ]:
build_summary = build_all(
    case_table, PROJECT_ROOT, MODEL_ROOT, TIF_FOLDER,
    overwrite=False, summary_csv=OUTPUT_FOLDER / "build_summary.csv",
)
print(build_summary["status"].value_counts())
build_summary.groupby(["variant", "split"]).size().unstack()

## Quick look

For one case per split: the 100 m DEM, the coarse 200 m / 400 m elevation and the subgrid
minimum / maximum elevation per cell.

In [ ]:
def plot_case(case_name):
    regular_dep = np.loadtxt(MODEL_ROOT / "regular_100m" / case_name / "sfincs.dep")
    panels = [("regular 100 m: dep", regular_dep)]

    for variant in ("subgrid_200m", "subgrid_400m"):
        sf = SfincsModel(root=str(MODEL_ROOT / variant / case_name), mode="r")
        sf.read()
        panels.append((f"{variant}: z_zmin", sf.subgrid["z_zmin"].values))
        panels.append((f"{variant}: z_zmax", sf.subgrid["z_zmax"].values))

    vmin = min(np.nanmin(p) for _, p in panels)
    vmax = max(np.nanmax(p) for _, p in panels)
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.2), constrained_layout=True)
    for axis, (title, grid) in zip(axes, panels):
        image = axis.imshow(grid, origin="lower", cmap="terrain", vmin=vmin, vmax=vmax)
        axis.set_title(title, fontsize=9)
        axis.set_xticks([])
        axis.set_yticks([])
    fig.colorbar(image, ax=axes.tolist(), label="Elevation [m]")
    fig.suptitle(case_name)
    plt.show()


for case_name in smoke_cases["case_name"]:
    plot_case(case_name)